## Search Engine with tools


In [1]:
# Arxiv Research
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper

C:\Users\Anshuman Pandey\AppData\Local\Temp\ipykernel_14376\3731374166.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun


In [2]:
import arxiv
from langchain_core.tools import tool

@tool
def arxiv_tool(query: str) -> str:
    """A wrapper around Arxiv.org. Useful for when you need to answer questions 
    about Physics, Mathematics, Computer Science, Quantitative Biology, Quantitative Finance, 
    Statistics, Electrical Engineering, and Economics from scientific articles on arxiv.org.
    Input should be an arXiv paper ID or a search query string.
    """
    try:
        # Force the modern client wrapper to handle secure transactions safely
        client = arxiv.Client()
        
        # Enforce your custom limit: top_k_results=1
        search = arxiv.Search(query=query, max_results=1)
        results = list(client.results(search))
        
        if not results:
            return f"No papers found matching the query: {query}"
            
        paper = results[0]
        full_payload = (
            f"Title: {paper.title}\n"
            f"Published: {paper.published.date()}\n"
            f"Abstract: {paper.summary}\n"
            f"Link: {paper.pdf_url}"
        )
        
        # Enforce your custom character limit: doc_content_chars_max=250
        return full_payload[:250]
        
    except Exception as e:
        return f"Error executing ArXiv lookup safely: {str(e)}"


In [3]:
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki_tool.run("Large Language Models")

'Page: Large language model\nSummary: A large language model (LLM) is a neural network trained on a vast amount of text for natural language processing tasks, especially language generation. LLMs can typically generate, summarize, translate, and analyz'

In [4]:
# TOOLS:
tools = [arxiv_tool, wiki_tool]

In [5]:
# Custom tool (RAG Tool)
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectordb = FAISS.from_documents(documents, OpenAIEmbeddings())
retriever = vectordb.as_retriever()
retriever

c:\Users\Anshuman Pandey\OneDrive\Desktop\LangChain\lang_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
USER_AGENT environment variable not set, consider setting it to identify your requests.


VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000017B5182FFA0>, search_kwargs={})

In [6]:
from langchain_core.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(retriever, "langsmith-search", "Search any information about langsmith")
retriever_tool

StructuredTool(name='langsmith-search', description='Search any information about langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000017B3D7084C0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000017B50FDC3A0>)

In [7]:
# Therefore,  tools are:
tools = [arxiv_tool, wiki_tool, retriever_tool]

## Run the tools with Agents and LLM Models

In [8]:
import os
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY)

In [9]:
# Create a LangSmith API in Settings > API Keys
# Make sure API key env var is set:
# import os; os.environ["LANGSMITH_API_KEY"] = "<your-api-key>"

os.environ["LANGCHAIN_TRACING_V2"] = "true"

from langsmith import Client

client = Client()
prompt = client.pull_prompt("hwchase17/openai-functions-agent", dangerously_pull_public_prompt=True)
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [10]:
#Create Agent
from langchain_classic.agents import create_tool_calling_agent

agent = create_tool_calling_agent(llm, tools, prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMes

In [11]:
## Agent Executer
from langchain_classic.agents import AgentExecutor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMe

In [14]:
agent_executor.invoke({"input":"Tell me about Langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langsmith-search` with `{'query': 'Langsmith information'}`


LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewEngineTraceDebugObserveLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics.LangSmith works with many frameworks and providers. Browse available integrations to connect your stack including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic AI, and more.Get startedCreate an accountSign up at smith.langchain.com (no credit card required).
You can log in with Google, GitHub, or email.Create an API keyGo to your Settings page → API Keys → 

{'input': 'Tell me about Langsmith',
 'output': 'LangSmith is a platform that provides observability and monitoring capabilities for large language models (LLMs) and other AI applications. It allows users to trace, debug, and optimize their LLMs, and provides features such as automated issue detection, diagnosis, and resolution. LangSmith also offers integrations with various frameworks and providers, including OpenAI, Anthropic, and Vercel AI SDK. Users can set up tracing, monitor performance, and collect feedback using the platform, and can also automate workflows and configure automations. Additionally, LangSmith provides a range of documentation and resources, including tutorials, API documentation, and a community forum. Overall, LangSmith aims to provide full visibility into LLM applications and help users improve their performance, quality, and reliability.'}

In [16]:
agent_executor.invoke({"input":"What's the paper 1706.03762 about?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv_tool` with `{'query': '1706.03762'}`


Title: Attention Is All You Need
Published: 2017-06-12
Abstract: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect th
Invoking: `arxiv_tool` with `{'query': '1706.03762'}`


Title: Attention Is All You Need
Published: 2017-06-12
Abstract: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect thThe paper "Attention Is All You Need" (arXiv:1706.03762) presents a novel neural network architecture for sequence-to-sequence tasks, such as machine translation. The proposed model, called Transformer, relies entirely on self-attention mechanisms to process input sequences, eliminating the need for recurrent neural networks (RNNs) or convoluti

{'input': "What's the paper 1706.03762 about?",
 'output': 'The paper "Attention Is All You Need" (arXiv:1706.03762) presents a novel neural network architecture for sequence-to-sequence tasks, such as machine translation. The proposed model, called Transformer, relies entirely on self-attention mechanisms to process input sequences, eliminating the need for recurrent neural networks (RNNs) or convolutional neural networks (CNNs). The Transformer model consists of an encoder and a decoder, each composed of a stack of identical layers. The encoder takes in a sequence of tokens (e.g., words or characters) and outputs a sequence of vectors, while the decoder generates the output sequence, one token at a time, based on the output vectors from the encoder. The self-attention mechanism allows the model to weigh the importance of different input elements relative to each other, enabling it to capture long-range dependencies and contextual relationships in the input sequence. The Transformer m